**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure3_signaling')


In [ ]:
# Import package
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import scipy.sparse as sp



In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))
adata

In [ ]:
sc.pl.umap(adata,color="celltype")

## Main 3 Additions — Figure 4 Preview Panels

- **Panel A** TLR4 sensing + TNF receptor machinery dotplot  
  9 myeloid subclusters × {TLR4, CD14, LY96(MD2), MYD88} + {TNFRSF1A, TNFRSF1B, TRADD, TRAF2}
- **Panel B** Pathway signature scores (sc.tl.score_genes)  
  REACTOME TLR4 CASCADE / HALLMARK TNFα-NF-κB / HALLMARK mTORC1 / REACTOME MyD88 CASCADE  
  → SLC7A5-high cluster highlighted: 두 자극 모두에 responsive한 transcriptional state 확인

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Main 3 · Panel A — TLR4 sensing + TNF receptor machinery dotplot
# 9 myeloid subclusters × 8 genes (two functional groups, visualised with dict var_names)
# standard_scale='var' → z-score per gene; highlights relative expression pattern
# ═══════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

SAVE_DIR = output_dir('02_SLC7A5_mono_Journal/05_Figure')
cluster_order = ['Classical_monocyte','Non-classical_monocyte','Special_monocyte','LAM','Kupffer_cell','mo_Kupffer_cell','cDC1','cDC2','mig_DC']

dp = sc.pl.dotplot(
    adata,
    var_names={
        'TLR4 sensing machinery' : ['TLR4', 'CD14', 'LY96', 'MYD88'],
        'TNF receptor machinery' : ['TNFRSF1A', 'TNFRSF1B', 'TRADD', 'TRAF2'],
    },
    groupby='celltype',
    categories_order=cluster_order,
    standard_scale='var',
    colorbar_title='Mean expr\n(z-scored)',
    size_title='% expressing',
    figsize=(9, 4),
    return_fig=True,
)

dp.savefig(f'{SAVE_DIR}/Fig3_PanelA_TLR4_TNF_dotplot.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Panel B · Step 1 — Download pathway gene sets via Enrichr API (no login)
#   Reactome_2022   →  TLR4 Cascade, MyD88 Cascade
#   MSigDB_Hallmark →  TNFα/NF-κB, mTORC1
# ═══════════════════════════════════════════════════════════════════════════

import requests

def load_enrichr_library(lib_name):
    url = f'https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={lib_name}'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    gene_sets = {}
    for line in r.text.strip().split('\n'):
        parts = line.split('\t')
        if len(parts) >= 3:
            gene_sets[parts[0]] = [g for g in parts[2:] if g]
    return gene_sets

print('Downloading MSigDB Hallmark 2020 ...')
hallmark = load_enrichr_library('MSigDB_Hallmark_2020')

print('Downloading Reactome 2022 ...')
reactome = load_enrichr_library('Reactome_2022')

# Pathway → gene list mapping (exact Enrichr key names)
PATHWAY_SETS = {
    'TLR4_CASCADE' : reactome['Toll Like Receptor 4 (TLR4) Cascade R-HSA-166016'],
    'TNFA_NFKB'    : hallmark['TNF-alpha Signaling via NF-kB'],
    'MTORC1'       : hallmark['mTORC1 Signaling'],
    'MYD88_CASCADE': reactome['MyD88 Cascade Initiated On Plasma Membrane R-HSA-975871'],
}

print('\nGene set coverage in adata:')
for name, genes in PATHWAY_SETS.items():
    n_present = sum(1 for g in genes if g in adata.var_names)
    print(f'  {name}: {n_present}/{len(genes)} genes')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Panel B · Step 2 — Pathway signature scoring
# sc.tl.score_genes = Seurat AddModuleScore equivalent
# Computes mean expression of gene set minus random background gene set
# ═══════════════════════════════════════════════════════════════════════════

SCORE_KEY_MAP = {
    'TLR4_CASCADE' : 'score_TLR4',
    'TNFA_NFKB'    : 'score_TNFA',
    'MTORC1'       : 'score_MTORC1',
    'MYD88_CASCADE': 'score_MYD88',
}

for set_name, score_name in SCORE_KEY_MAP.items():
    genes_in_data = [g for g in PATHWAY_SETS[set_name] if g in adata.var_names]
    sc.tl.score_genes(adata, gene_list=genes_in_data, score_name=score_name)
    print(f'  {score_name}: {len(genes_in_data)} genes used')

print('\nScore distributions:')
adata.obs[list(SCORE_KEY_MAP.values())].describe().round(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Panel B · Step 3 — Violin plots: pathway scores per myeloid subcluster
# SLC7A5-high cluster (highest mean SLC7A5 expression) highlighted in red
# ═══════════════════════════════════════════════════════════════════════════

import scipy.sparse as sp
import seaborn as sns
import matplotlib.patches as mpatches

cluster_order = ['Classical_monocyte','Non-classical_monocyte','Special_monocyte','LAM','Kupffer_cell','mo_Kupffer_cell','cDC1','cDC2','mig_DC']

# ── Identify SLC7A5-high subcluster ──────────────────────────────────────────
X_mat = adata.X.toarray() if sp.issparse(adata.X) else adata.X
slc7a5_idx   = adata.var_names.get_loc('SLC7A5')
slc7a5_mean  = (
    adata.obs.assign(SLC7A5_expr=X_mat[:, slc7a5_idx])
    .groupby('celltype')['SLC7A5_expr']
    .mean()
)
slc7a5_high  = slc7a5_mean.idxmax()
print(f'SLC7A5-high cluster: {slc7a5_high}  (mean = {slc7a5_mean[slc7a5_high]:.3f})')
print(slc7a5_mean.sort_values(ascending=False).round(3).to_string())

# ── Build long-form dataframe ─────────────────────────────────────────────────
score_cols   = ['score_TLR4', 'score_TNFA', 'score_MTORC1', 'score_MYD88']
panel_titles = [
    'REACTOME\nTLR4 Cascade',
    'HALLMARK\nTNFα / NF-κB',
    'HALLMARK\nmTORC1 Signaling',
    'REACTOME\nMyD88 Cascade',
]
plot_df = adata.obs[['celltype'] + score_cols].copy()

# ── Palette: SLC7A5-high in red, others grey ──────────────────────────────────
palette = {c: '#D62728' if c == slc7a5_high else '#AAAAAA' for c in cluster_order}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col, title in zip(axes, score_cols, panel_titles):
    sns.violinplot(
        data=plot_df,
        x='celltype',
        y=col,
        order=cluster_order,
        palette=palette,
        inner='box',
        cut=0,
        linewidth=0.7,
        ax=ax,
    )
    ax.set_title(title, fontsize=10, pad=8, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Signature score' if ax is axes[0] else '', fontsize=9)
    ax.set_xticklabels(cluster_order, rotation=45, ha='right', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['right', 'top']].set_visible(False)

legend_handles = [
    mpatches.Patch(color='#D62728', label=f'SLC7A5-high ({slc7a5_high})'),
    mpatches.Patch(color='#AAAAAA', label='Other subclusters'),
]
fig.legend(
    handles=legend_handles,
    loc='upper right',
    bbox_to_anchor=(0.99, 0.99),
    fontsize=9,
    frameon=False,
)

fig.suptitle(
    'Panel B — Pathway signature scores across myeloid subclusters',
    y=1.02, fontsize=12, fontweight='bold',
)
fig.tight_layout()
plt.savefig(f'{SAVE_DIR}/Fig3_PanelB_pathway_scores.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Panel B · Step 4 — Statistical annotation: mono5 vs. all other clusters
# Mann-Whitney U test (one-sided: mono5 > rest) per pathway score
# * p<0.05  ** p<0.01  *** p<0.001  ns = not significant
# ═══════════════════════════════════════════════════════════════════════════

from scipy.stats import mannwhitneyu
import scipy.sparse as sp
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SAVE_DIR       = output_dir('02_SLC7A5_mono_Journal/05_Figure')
TARGET_CLUSTER = 'Special_monocyte'
score_cols     = ['score_TLR4', 'score_TNFA', 'score_MTORC1', 'score_MYD88']
panel_titles   = [
    'REACTOME\nTLR4 Cascade',
    'HALLMARK\nTNFα / NF-κB',
    'HALLMARK\nmTORC1 Signaling',
    'REACTOME\nMyD88 Cascade',
]

def pval_to_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

plot_df = adata.obs[['celltype'] + score_cols].copy()

# ── Statistical tests ─────────────────────────────────────────────────────
print(f'Mann-Whitney U  ({TARGET_CLUSTER} > rest, one-sided)')
print(f'{"Score":<20} {"U-stat":>10} {"p-value":>12} {"Sig":>5}')
print('-' * 52)
sig_results = {}
for col in score_cols:
    tgt   = plot_df.loc[plot_df['celltype'] == TARGET_CLUSTER, col].values
    rest  = plot_df.loc[plot_df['celltype'] != TARGET_CLUSTER, col].values
    u, p  = mannwhitneyu(tgt, rest, alternative='greater')
    stars = pval_to_stars(p)
    sig_results[col] = (u, p, stars)
    print(f'{col:<20} {u:>10.0f} {p:>12.4e} {stars:>5}')

# ── SLC7A5-high cluster identification ───────────────────────────────────
X_mat = adata.X.toarray() if sp.issparse(adata.X) else adata.X
slc7a5_idx  = adata.var_names.get_loc('SLC7A5')
slc7a5_mean = (
    adata.obs.assign(SLC7A5_expr=X_mat[:, slc7a5_idx])
    .groupby('celltype')['SLC7A5_expr']
    .mean()
)
slc7a5_high = slc7a5_mean.idxmax()
palette = {c: '#D62728' if c == slc7a5_high else '#AAAAAA' for c in cluster_order}

# ── Violin plot with significance stars ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col, title in zip(axes, score_cols, panel_titles):
    sns.violinplot(
        data=plot_df,
        x='celltype',
        y=col,
        order=cluster_order,
        palette=palette,
        inner='box',
        cut=0,
        linewidth=0.7,
        ax=ax,
    )
    ax.set_title(title, fontsize=10, pad=8, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Signature score' if ax is axes[0] else '', fontsize=9)
    ax.set_xticklabels(cluster_order, rotation=45, ha='right', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['right', 'top']].set_visible(False)

    # significance star above mono5
    _, p_val, stars = sig_results[col]
    x_pos  = cluster_order.index(TARGET_CLUSTER)
    y_top  = plot_df.loc[plot_df['celltype'] == TARGET_CLUSTER, col].max()
    y_span = ax.get_ylim()[1] - ax.get_ylim()[0]
    star_color = '#D62728' if stars != 'ns' else '#888888'
    ax.text(x_pos, y_top + y_span * 0.05, stars,
            ha='center', va='bottom', fontsize=13,
            fontweight='bold', color=star_color)

legend_handles = [
    mpatches.Patch(color='#D62728', label=f'SLC7A5-high ({slc7a5_high})'),
    mpatches.Patch(color='#AAAAAA', label='Other subclusters'),
]
fig.legend(handles=legend_handles, loc='upper right',
           bbox_to_anchor=(0.99, 0.99), fontsize=9, frameon=False)

fig.suptitle(
    'Panel B — Pathway signature scores  |  * = mono5 > rest  (Mann-Whitney U, one-sided)',
    y=1.02, fontsize=11, fontweight='bold',
)
fig.tight_layout()
plt.savefig(f'{SAVE_DIR}/Fig3_PanelB_pathway_scores_stats.pdf',
            bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {SAVE_DIR}/Fig3_PanelB_pathway_scores_stats.pdf')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# mono5 vs. mono1 / mono5 vs. mono4  pairwise Mann-Whitney U (one-sided)
# ═══════════════════════════════════════════════════════════════════════════

from scipy.stats import mannwhitneyu
import pandas as pd

score_cols = ['score_TLR4', 'score_TNFA', 'score_MTORC1', 'score_MYD88']
comparisons = [('Special_monocyte', 'Classical_monocyte'), ('Special_monocyte', 'LAM')]

def pval_to_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

plot_df = adata.obs[['celltype'] + score_cols].copy()

rows = []
for col in score_cols:
    for a, b in comparisons:
        vals_a = plot_df.loc[plot_df['celltype'] == a, col].values
        vals_b = plot_df.loc[plot_df['celltype'] == b, col].values
        u, p   = mannwhitneyu(vals_a, vals_b, alternative='greater')
        rows.append({
            'Score'      : col,
            'Comparison' : f'{a} > {b}',
            'n_a'        : len(vals_a),
            'n_b'        : len(vals_b),
            'U-statistic': round(u, 1),
            'p-value'    : p,
            'Sig'        : pval_to_stars(p),
        })

result_df = pd.DataFrame(rows)
result_df['p-value'] = result_df['p-value'].map(lambda x: f'{x:.4e}')
display(result_df.set_index(['Score', 'Comparison']))